
# Multiscale correlation-matrix analysis (LRG pipeline)

This notebook demonstrates how to take one or more fMRI correlation matrices, apply
band-aware noise filtering, and run a Laplacian renormalisation group (LRG) style
multiscale clustering workflow. Use it as a template for comparing spatial clusters
across frequency bands or imaging contrasts.



## Requirements and inputs
- A square correlation matrix for each contrast/band you want to analyse.
- (Optional) Raw time-series data if you want to recreate the correlation matrix from scratch using bandpass filters.
- The utilities in `multifunbrain.analysis` and `multifunbrain.core` for Laplacian diffusion, clustering, and filtering.

All paths are relative to the repository root so the notebook remains portable.


In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np

from multifunbrain.analysis.corrnet import compute_correlation_matrix
from multifunbrain.analysis.corrmatrix import (
    load_correlation_matrix,
    prepare_correlation_matrix,
    marchenko_pastur_denoise,
    hierarchical_partitions_from_corr,
    adjusted_rand_index,
    compare_partition_sets,
)
from multifunbrain.core import band_filter

## Helper functions
All helper functions have been promoted to the `multifunbrain.analysis.corrmatrix`
module and are imported above. See the module docstring for full API details.

In [ ]:
# All helper functions now live in multifunbrain.analysis.corrmatrix
# (imported in the cell above). The original inline definitions have been
# removed to keep the notebook in sync with the package.


## Example data
Below we synthesise two band-limited correlation matrices from toy signals. Replace
`corr_band_a` and `corr_band_b` with your own matrices to reuse the pipeline.


In [ ]:

rng = np.random.default_rng(42)

n_regions = 32
n_timepoints = 600
fs = 1 / 1.2

# Base multichannel signal with shared latent trends
base_signal = rng.normal(size=(n_regions, n_timepoints))
trend = np.sin(np.linspace(0, 6 * np.pi, n_timepoints))
base_signal += 0.2 * trend

# Create two contrasted bands
slow5 = band_filter(base_signal, low=0.01, high=0.027, fs=fs)
slow4 = band_filter(base_signal, low=0.027, high=0.073, fs=fs)

corr_band_a = compute_correlation_matrix(slow5)
corr_band_b = compute_correlation_matrix(slow4)

# Optional denoising step
corr_band_a = marchenko_pastur_denoise(corr_band_a, gamma=0.6)
corr_band_b = marchenko_pastur_denoise(corr_band_b, gamma=0.6)



## Run diffusion/LRG clustering
Choose the diffusion scales (`tau_values`) to explore the multiscale structure of
the correlation networks. The partitions keep track of dendrogram thresholds and
labels for downstream comparison.


In [ ]:

tau_values = np.logspace(-2, 1, 6)

partitions_a = hierarchical_partitions_from_corr(corr_band_a, tau_values, edge_threshold=0.0)
partitions_b = hierarchical_partitions_from_corr(corr_band_b, tau_values, edge_threshold=0.0)

for part in partitions_a[:2]:
    print(f"τ={part['tau']:.3f} -> {len(np.unique(part['partition']))} clusters (band A)")
for part in partitions_b[:2]:
    print(f"τ={part['tau']:.3f} -> {len(np.unique(part['partition']))} clusters (band B)")



## Cross-matrix clustering agreement
We use the Adjusted Rand Index (ARI) to quantify how similar the community
assignments are between bands or contrasts at different diffusion scales.


In [ ]:

comparison = compare_partition_sets(partitions_a, partitions_b)

for row in comparison:
    if row["tau_a"] == row["tau_b"]:
        print(f"τ={row['tau_a']:.3f} | ARI={row['ari']:.3f}")



## Next steps
- Replace the synthetic matrices with your real band/contrast correlation matrices.
- Tune the diffusion scales to match the temporal resolution of your experiment.
- Persist `partitions_*` to disk (e.g., via `pickle` or JSON) to compare many
  subjects or contrasts outside the notebook.
